In [2]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
     

import torchvision
import torchvision.transforms as transforms

from models import *
use_gpu = torch.cuda.is_available()
batch_size = 128
model_name = "vgg16_quant_iter1"
model = VGG16_quant()

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])

In [3]:
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))

trainloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))


testloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

In [11]:
print_freq = 390 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)

        input = input.cuda(non_blocking=True)
        target = target.cuda(non_blocking=True)

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input = input.cuda(non_blocking=True)
            target = target.cuda(non_blocking=True)

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

        

In [28]:
def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [45, 90, 180]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1     

In [29]:
import os
import glob
import shutil
import torch

def save_checkpoint(state, is_best, fdir, best_prec):
    os.makedirs(fdir, exist_ok=True)
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)

    if is_best:
        # Remove previous best checkpoint
        for old_file in glob.glob(os.path.join(fdir, 'model_best_prec=*.pth.tar')):
            os.remove(old_file)

        # Save new best checkpoint with accuracy in filename
        best_path = os.path.join(fdir, f'model_best_prec={best_prec:.3f}.pth.tar')
        shutil.copyfile(filepath, best_path)


In [30]:
model_name = "vgg16_qat"
cudnn.benchmark = True
criterion = nn.CrossEntropyLoss().cuda()

def MFHPO_LR_WD(learning_rate, weight_decay, max_epochs):
    model = VGG16_quant()
    best_prec = 0
    model.cuda()
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=weight_decay)
    model_name_new = model_name + f"_lr={learning_rate}_wd={weight_decay}"
    fdir = 'results_MFHPO/'+str(model_name_new)
    if not os.path.exists(fdir):
        os.makedirs(fdir)
    for epoch in range(0, max_epochs):
        adjust_learning_rate(optimizer, epoch)
    
        train(trainloader, model, criterion, optimizer, epoch)
        
        # evaluate on test set
        print("Validation starts")
        prec = validate(testloader, model, criterion)
    
        # remember best precision and save checkpoint
        is_best = prec > best_prec
        best_prec = max(prec,best_prec)
        print('best acc: {:1f}'.format(best_prec))
        save_checkpoint({
            'epoch': epoch + 1,
            'state_dict': model.state_dict(),
            'best_prec': best_prec,
            'optimizer': optimizer.state_dict(),
        }, is_best, fdir, best_prec)
    return best_prec

In [31]:
import pandas as pd
model_name = "vgg16_qat"
cudnn.benchmark = True
criterion = nn.CrossEntropyLoss().cuda()
learning_rates = [
    0.0001, 0.0005, 0.001, 0.0025,
    0.005, 0.01, 0.02, 0.05, 
]

weight_decays = [
    0.0, 1e-5, 1e-4, 5e-4, 2e-3
]
max_epochs = 60
best_prec = 0

results = []
best_lr = None
best_wd = None
global_best_prec = 0

for lr in learning_rates:
    for wd in weight_decays:
        best_prec = MFHPO_LR_WD(lr, wd, max_epochs)
        if best_prec >= global_best_prec:
            global_best_prec = best_prec
            best_lr = lr
            best_wd = wd
        results.append({
            'lr': lr,
            'weight_decay': wd,
            'best_prec': best_prec
        })

        # Save progress after every trial
        df = pd.DataFrame(results)
        df.to_csv('sweep_1_results.csv', index=False)

# Sort by highest accuracy
df = df.sort_values(by='best_prec', ascending=False)
df = df.reset_index(drop=True)


display(df)
            

Epoch: [0][0/391]	Time 0.203 (0.203)	Data 0.056 (0.056)	Loss 2.5288 (2.5288)	Prec 13.281% (13.281%)
Epoch: [0][390/391]	Time 0.038 (0.030)	Data 0.000 (0.000)	Loss 1.5967 (1.8559)	Prec 38.750% (30.138%)
Validation starts
Test: [0/79]	Time 0.041 (0.041)	Loss 1.5694 (1.5694)	Prec 43.750% (43.750%)
 * Prec 38.850% 
best acc: 38.850000
Epoch: [1][0/391]	Time 0.120 (0.120)	Data 0.075 (0.075)	Loss 1.6858 (1.6858)	Prec 35.938% (35.938%)
Epoch: [1][390/391]	Time 0.020 (0.029)	Data 0.000 (0.001)	Loss 1.4279 (1.5776)	Prec 52.500% (41.082%)
Validation starts
Test: [0/79]	Time 0.039 (0.039)	Loss 1.4300 (1.4300)	Prec 46.094% (46.094%)
 * Prec 46.270% 
best acc: 46.270000
Epoch: [2][0/391]	Time 0.094 (0.094)	Data 0.068 (0.068)	Loss 1.5521 (1.5521)	Prec 43.750% (43.750%)
Epoch: [2][390/391]	Time 0.023 (0.029)	Data 0.000 (0.001)	Loss 1.4148 (1.4645)	Prec 45.000% (45.916%)
Validation starts
Test: [0/79]	Time 0.037 (0.037)	Loss 1.4007 (1.4007)	Prec 43.750% (43.750%)
 * Prec 49.390% 
best acc: 49.390000
E

,lr,weight_decay,best_prec
0,0.0050,0.00200,92.16
1,0.0200,0.00050,92.11
2,0.0100,0.00200,92.10
3,0.0500,0.00050,91.36
4,0.0100,0.00050,91.30
5,0.0500,0.00010,91.03
6,0.0050,0.00050,90.88
7,0.0200,0.00010,90.52
8,0.0025,0.00200,90.47
9,0.0100,0.00010,90.20


In [32]:
display(df)

,lr,weight_decay,best_prec
0,0.0050,0.00200,92.16
1,0.0200,0.00050,92.11
2,0.0100,0.00200,92.10
3,0.0500,0.00050,91.36
4,0.0100,0.00050,91.30
5,0.0500,0.00010,91.03
6,0.0050,0.00050,90.88
7,0.0200,0.00010,90.52
8,0.0025,0.00200,90.47
9,0.0100,0.00010,90.20


In [33]:
print(best_lr)
print(best_wd)

0.005
0.002


## Tournmanent Scheduling:
(Utilizing best lr and wd discovered from MF-HPO lr/wd)
### Round 1 - 
- 16 VGG16_quant Initializations.
- 30 Epochs per initialization.
- Worst 8 models (by test validation) are dropped.
### Round 2 -
- 8 Best models from Round 1.
- 40 Additional epochs per model. (70 epochs total per model)
- Worst 4 models are dropped.
### Round 3 -
- 4 Best models from Round 2.
- 50 Additional epochs per model. (120 epochs total per model)
- Worst 2 models are dropped.
### Round 4 - 
-  2 Best models from round 3.
-  90 additional epochs per model. (210 epochs total per model)
-  Worst model dropped.

### Winning model receives guided training.

In [34]:
def keep_best(d, n):
    return dict(sorted(d.items(), key=lambda x: x[1], reverse=True)[:n])

In [35]:
def tournament_load_train(model_id, max_epochs):
    fdir = f"results_tournament/vgg16_qat_{model_id}"
    filepath = os.path.join(fdir, "checkpoint.pth")

    checkpoint = torch.load(
        filepath,
        map_location="cuda",
        weights_only=True
    )

    model = VGG16_quant().cuda()
    model.load_state_dict(checkpoint["state_dict"])

    # Placeholder LR; replaced by saved optimizer state
    optimizer = torch.optim.SGD(model.parameters(), lr=0.0)
    optimizer.load_state_dict(checkpoint["optimizer"])

    start_epoch = checkpoint["epoch"]
    best_prec = checkpoint["best_prec"]

    for epoch in range(start_epoch, start_epoch+max_epochs):
        adjust_learning_rate(optimizer, epoch)
    
        train(trainloader, model, criterion, optimizer, epoch)
        
        # evaluate on test set
        print("Validation starts")
        prec = validate(testloader, model, criterion)
    
        # remember best precision and save checkpoint
        is_best = prec > best_prec
        best_prec = max(prec,best_prec)
        print('best acc: {:1f}'.format(best_prec))
        save_checkpoint({
            'epoch': epoch + 1,
            'state_dict': model.state_dict(),
            'best_prec': best_prec,
            'optimizer': optimizer.state_dict(),
        }, is_best, fdir, best_prec)
        
    return best_prec

In [36]:
learning_rate = best_lr
weight_decay = best_wd
contenders = {}
round_1_epochs = 30
round_2_epochs = 40
round_3_epochs = 50
round_4_epochs = 90
model_name = "vgg16_qat"
# Round 1:
for model_id in range(16):
    model = VGG16_quant()
    best_prec = 0
    model.cuda()
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=weight_decay)
    model_name_new = model_name + f"_{model_id}"
    fdir = 'results_tournament/'+str(model_name_new)
    if not os.path.exists(fdir):
        os.makedirs(fdir)
    for epoch in range(0, round_1_epochs):
        adjust_learning_rate(optimizer, epoch)
    
        train(trainloader, model, criterion, optimizer, epoch)
        
        # evaluate on test set
        print("Validation starts")
        prec = validate(testloader, model, criterion)
    
        # remember best precision and save checkpoint
        is_best = prec > best_prec
        best_prec = max(prec,best_prec)
        print('best acc: {:1f}'.format(best_prec))
        save_checkpoint({
            'epoch': epoch + 1,
            'state_dict': model.state_dict(),
            'best_prec': best_prec,
            'optimizer': optimizer.state_dict(),
        }, is_best, fdir, best_prec)
    contenders[model_id] = best_prec

#Eliminate Worst 8:
contenders = keep_best(contenders, 8)

#Round 2:
for model_id in contenders:
    best_prec = tournament_load_train(model_id, round_2_epochs)
    contenders[model_id] = best_prec

contenders = keep_best(contenders, 4)

#Round 3:
for model_id in contenders:
    best_prec = tournament_load_train(model_id, round_3_epochs)
    contenders[model_id] = best_prec

contenders = keep_best(contenders, 2)

#Round 4:
for model_id in contenders:
    best_prec = tournament_load_train(model_id, round_4_epochs)
    contenders[model_id] = best_prec

contenders = keep_best(contenders, 1)

Epoch: [0][0/391]	Time 0.073 (0.073)	Data 0.048 (0.048)	Loss 2.5948 (2.5948)	Prec 8.594% (8.594%)
Epoch: [0][390/391]	Time 0.023 (0.029)	Data 0.000 (0.000)	Loss 1.3712 (1.5977)	Prec 51.250% (41.480%)
Validation starts
Test: [0/79]	Time 0.044 (0.044)	Loss 2.0440 (2.0440)	Prec 44.531% (44.531%)
 * Prec 40.610% 
best acc: 40.610000
Epoch: [1][0/391]	Time 0.087 (0.087)	Data 0.065 (0.065)	Loss 1.2993 (1.2993)	Prec 50.781% (50.781%)
Epoch: [1][390/391]	Time 0.020 (0.028)	Data 0.000 (0.001)	Loss 0.9595 (1.1320)	Prec 62.500% (59.102%)
Validation starts
Test: [0/79]	Time 0.042 (0.042)	Loss 1.0036 (1.0036)	Prec 60.156% (60.156%)
 * Prec 62.410% 
best acc: 62.410000
Epoch: [2][0/391]	Time 0.091 (0.091)	Data 0.068 (0.068)	Loss 0.8788 (0.8788)	Prec 67.188% (67.188%)
Epoch: [2][390/391]	Time 0.020 (0.029)	Data 0.000 (0.000)	Loss 0.8298 (0.9523)	Prec 71.250% (66.392%)
Validation starts
Test: [0/79]	Time 0.042 (0.042)	Loss 1.0306 (1.0306)	Prec 64.844% (64.844%)
 * Prec 64.790% 
best acc: 64.790000
Epo

In [37]:
print(contenders)

{0: 92.61}
